# 05 · Atendimentos da Vulto: raio-x

Fonte: `atendimentos.csv` (export da ferramenta de atendimento: WhatsApp, Instagram e e-mail, gerado por `padronizar.py`; as 17 linhas duplicadas idênticas já foram removidas).

Objetivo: **entender** os atendimentos, sem decidir nada. A gerente de atendimento quer esses atendimentos **ligados ao cliente e ao pedido certo**.

> Antes de commitar: *Kernel → Restart & Clear Output* (as tabelas têm CPF, e-mail e telefone).

In [ ]:
from pathlib import Path

import pandas as pd

# raiz do repositório, de onde quer que o notebook tenha sido aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "migracao_vulto").is_dir())
STAGING = RAIZ / "staging_vulto"

# mostra todas as colunas nas tabelas
pd.set_option("display.max_columns", None)

In [ ]:
atendimentos = pd.read_csv(STAGING / "atendimentos.csv", dtype=str)
clientes = pd.read_csv(STAGING / "clientes_clientes.csv", dtype=str)
vendas = pd.read_csv(STAGING / "banco_vendas.csv", dtype=str)

## 1. O que é cada linha?

Um atendimento: a conversa inteira com o cliente, com as mensagens do atendente dentro do texto (`[Atendente]: ...`).

In [ ]:
print(atendimentos.shape, "| período:", atendimentos["data_hora"].min()[:10], "a", atendimentos["data_hora"].max()[:10])
print("conversas com resposta do atendente no texto:", atendimentos["mensagem"].fillna("").str.contains(r"\[Atendente\]").sum())
atendimentos.head(3)

## 2. O que significa cada coluna?

In [ ]:
for coluna in ["canal", "status", "assunto", "atendente", "tags", "contato_tipo", "pedido_tipo"]:
    print(coluna, "|", atendimentos[coluna].value_counts(dropna=False).to_dict())

## 3 e 4. Problemas e ligações (cliente e pedido)

In [ ]:
# contato → cliente: telefone e e-mail podem ser procurados no cadastro; Instagram não
telefones, emails = set(clientes["Tel"].dropna()), set(clientes["E-mail"].dropna().str.lower())
def acha(tipo, valor):
    if pd.isna(tipo):
        return "sem contato"
    if tipo == "instagram":
        return "Instagram (sem como achar)"
    return "achou" if (valor in telefones if tipo == "telefone" else valor.lower() in emails) else "não achou"
pd.crosstab(atendimentos["contato_tipo"].fillna("(vazio)"), [acha(t, v) for t, v in zip(atendimentos["contato_tipo"], atendimentos["contato_valor"])])

In [ ]:
# pedido → venda: número do site/app ou número solto (id interno)
codigos = atendimentos["pedido_codigo"].dropna()
print("com pedido:", len(codigos), "| sem pedido:", atendimentos["pedido_codigo"].isna().sum())
print("acham pelo número do site/app:", codigos.isin(vendas["pedido_externo"]).sum(),
      "| acham pelo id interno:", codigos.isin(vendas["id"]).sum(),
      "| não acham:", (~codigos.isin(set(vendas["pedido_externo"].dropna()) | set(vendas["id"]))).sum())
atendimentos.loc[atendimentos["assunto"] == "spam", "mensagem"].str[:60].head()

## Desafios da revisão

In [ ]:
# os números soltos acham a venda certa? a venda deveria ser antes do atendimento, e não muito antes
soltos = atendimentos[atendimentos["pedido_tipo"] == "numero"].dropna(subset=["pedido_codigo"]).merge(
    vendas[["id", "criado_em"]], left_on="pedido_codigo", right_on="id", suffixes=("", "_venda"))
dias = (pd.to_datetime(soltos["data_hora"]) - pd.to_datetime(soltos["criado_em"])).dt.days
print("números soltos que acham uma venda:", len(soltos), "| venda até 60 dias antes do atendimento:", dias.between(0, 60).sum(),
      "| venda depois do atendimento:", (dias < 0).sum(), "| mais de 60 dias antes:", (dias > 60).sum(),
      "| mediana:", dias[dias.between(0, 60)].median(), "dias")

## 5. O pedido também identifica o cliente

O pedido citado tem um `cliente_ref`: quando a venda é achada, o dono do pedido é um segundo caminho até o cliente, além do contato. Antes de cruzar os dois, três coisas escondem pedidos:
- o cliente escreveu o número na mensagem, mas a coluna `pedido` veio vazia;
- número solto de 7 dígitos começando em 7, que é o pedido do app sem o `APP-`;
- número com os dígitos invertidos (`874101` → `S101478`).

In [ ]:
import re
import unicodedata

# código de pedido escrito pelo cliente (só a primeira fala, antes da resposta do atendente)
CODIGO = r"\b(S\d{5,7}|APP-\d+|#\s?\d{3,7}|pedido\s*\d{3,8}|\d{7})\b"
primeira_fala = atendimentos["mensagem"].fillna("").str.split(r"\[").str[0]
no_texto = (primeira_fala.str.extract(CODIGO, flags=re.IGNORECASE)[0]
            .str.upper().str.replace("PEDIDO", "").str.replace("#", "").str.strip())
print("pedido vazio na coluna, mas escrito no texto:", (atendimentos["pedido_codigo"].isna() & no_texto.notna()).sum())

# número solto de 7 dígitos começando em 7: é o pedido do app sem o prefixo?
soltos7 = atendimentos.loc[atendimentos["pedido_tipo"] == "numero", "pedido_codigo"].dropna()
soltos7 = soltos7[soltos7.str.fullmatch(r"7\d{6}")]
print("números soltos com cara de app:", len(soltos7), "| batem com APP-<número>:", ("APP-" + soltos7).isin(vendas["pedido_externo"]).sum(),
      "| batem com o id interno:", soltos7.isin(vendas["id"]).sum())

In [ ]:
# liga cada atendimento a uma venda: código da coluna ou do texto; app sem prefixo; número solto só se a venda
# for de até 60 dias antes (seção "Desafios"); por último, dígitos invertidos, com a mesma regra dos 60 dias
# código repetido (V5): fica a última linha, a paga, que é a que migra
externos = vendas.dropna(subset=["pedido_externo"]).drop_duplicates("pedido_externo", keep="last").set_index("pedido_externo")
por_id = vendas.set_index("id")
atendimentos["codigo"] = atendimentos["pedido_codigo"].fillna(no_texto)
atendimentos["codigo_de"] = atendimentos["pedido_codigo"].notna().map({True: "coluna", False: "texto"}).where(atendimentos["codigo"].notna())

def ate_60_dias_antes(atendimento, venda):
    return 0 <= (pd.to_datetime(atendimento["data_hora"]) - pd.to_datetime(venda["criado_em"])).days <= 60

def acha_venda(atendimento):
    codigo = atendimento["codigo"]
    if pd.isna(codigo):
        return None, "sem código"
    if re.fullmatch(r"7\d{6}", codigo):
        codigo = "APP-" + codigo
    if codigo in externos.index:
        return externos.loc[codigo, "id"], "código" if codigo == atendimento["codigo"] else "app sem prefixo"
    if codigo.isdigit() and codigo in por_id.index and ate_60_dias_antes(atendimento, por_id.loc[codigo]):
        return codigo, "número solto"
    invertido = re.sub(r"\D", "", codigo)[::-1]
    for candidato in ("S" + invertido, "APP-" + invertido):
        if candidato in externos.index and ate_60_dias_antes(atendimento, externos.loc[candidato]):
            return externos.loc[candidato, "id"], "dígitos invertidos"
    return None, "código não acha"

ligacao = atendimentos.apply(acha_venda, axis=1)
atendimentos["id_venda"], atendimentos["como_achou_venda"] = ligacao.str[0], ligacao.str[1]
pd.crosstab(atendimentos["como_achou_venda"], atendimentos["codigo_de"].fillna("-"), margins=True)

In [ ]:
# os invertidos acham a venda certa? sem a regra dos 60 dias, a data da venda deveria ficar perto da do atendimento
nao_acham = atendimentos[atendimentos["como_achou_venda"].isin(["código não acha", "dígitos invertidos"])]
invertidos = nao_acham["codigo"].str.replace(r"\D", "", regex=True).str[::-1]
candidatos = pd.concat([("S" + invertidos)[("S" + invertidos).isin(externos.index)], ("APP-" + invertidos)[("APP-" + invertidos).isin(externos.index)]])
dias = (pd.to_datetime(nao_acham.loc[candidatos.index, "data_hora"]) - pd.to_datetime(externos.loc[candidatos, "criado_em"].values)).dt.days
print("códigos que não acham venda:", len(nao_acham), "| invertidos acham:", len(candidatos), "| venda até 60 dias antes:", dias.between(0, 60).sum(),
      "| mediana:", dias[dias.between(0, 60)].median(), "dias | faixa dos códigos do site:", externos.index[externos.index.str.startswith("S")].min(), "a",
      externos.index[externos.index.str.startswith("S")].max())

In [ ]:
# cliente por dois caminhos: o contato de quem escreveu e o dono do pedido (o cliente_ref da venda)
import sys
sys.path.insert(0, str(RAIZ))
from src.utils.cpf import cpf_valido

cpf_por_tel = dict(zip(clientes["Tel"], clientes["cpf"]))
cpf_por_email = dict(zip(clientes["E-mail"].str.lower(), clientes["cpf"]))
cpfs = set(clientes["cpf"].dropna())

# regra da J1 (registro): o CPF é a identidade; sem CPF, o e-mail identifica um cliente sem conta;
# o telefone só serve para achar o CPF no cadastro
def cpf_do_cliente(tipo, valor):
    if pd.isna(valor):
        return None
    if tipo == "email":
        return cpf_por_email.get(valor.lower())
    if tipo == "telefone":
        return cpf_por_tel.get(valor)
    if tipo == "cpf":
        return valor
    if tipo == "numero":  # J1b: o cadastro decide primeiro (CPF ou telefone); fora dele, vira CPF se o dígito fecha
        if valor in cpfs:
            return valor
        return cpf_por_tel.get("+55" + valor) or (valor if cpf_valido(valor) else None)
    return None

def identidade(tipo, valor):
    cpf = cpf_do_cliente(tipo, valor)
    if pd.notna(cpf):
        return cpf
    return valor.lower() if tipo == "email" and pd.notna(valor) else None

vendas["identidade"] = [identidade(t, v) for t, v in zip(vendas["cliente_tipo"], vendas["cliente_valor"])]
compradores_email = set(vendas["identidade"].dropna()[lambda i: i.str.contains("@")])

def cliente_pelo_contato(tipo, valor):
    cpf = cpf_do_cliente(tipo, valor)
    if pd.notna(cpf):
        return cpf
    return valor.lower() if tipo == "email" and valor.lower() in compradores_email else None

atendimentos["cliente_pelo_contato"] = [cliente_pelo_contato(t, v) for t, v in zip(atendimentos["contato_tipo"], atendimentos["contato_valor"])]
atendimentos["cliente_pelo_pedido"] = atendimentos["id_venda"].map(vendas.set_index("id")["identidade"])

# ponte pelo atendimento: um lado dá o CPF e o outro só um e-mail -> o e-mail é dessa pessoa
# (quando os dois lados têm CPF, eles sempre concordam, então contato e pedido falam da mesma pessoa)
def e_cpf(i):
    return isinstance(i, str) and "@" not in i
def e_email(i):
    return isinstance(i, str) and "@" in i
contato, pedido = atendimentos["cliente_pelo_contato"], atendimentos["cliente_pelo_pedido"]
# o e-mail do contato vale mesmo que a pessoa nunca tenha comprado com ele (contato vazio)
email_do_contato = (contato.map(e_email) | (atendimentos["contato_tipo"].eq("email") & contato.isna())) & pedido.map(e_cpf)
email_do_pedido = pedido.map(e_email) & contato.map(e_cpf)
atendimentos["ponte"] = email_do_contato | email_do_pedido
ponte = {**dict(zip(atendimentos.loc[email_do_contato, "contato_valor"].str.lower(), pedido[email_do_contato])),
         **dict(zip(pedido[email_do_pedido], contato[email_do_pedido]))}
vendas["identidade"] = vendas["identidade"].map(lambda i: ponte.get(i, i) if isinstance(i, str) else i)
print("atendimentos-ponte:", atendimentos["ponte"].sum(), "| pares e-mail → CPF:", len(ponte),
      "| vendas que passam do e-mail para o CPF:", vendas["cliente_valor"].str.lower().isin(ponte.keys()).sum())

venda = vendas.set_index("id")
atendimentos["cliente_pelo_pedido"] = atendimentos["id_venda"].map(venda["identidade"])
atendimentos["cliente_pelo_contato"] = atendimentos["cliente_pelo_contato"].map(lambda i: ponte.get(i, i) if isinstance(i, str) else i)
# "NAO INFORMADO" e "consumidor" são venda sem cliente, não um cliente
ref = venda["cliente_ref"].where(~venda["cliente_ref"].str.lower().isin(["nao informado", "consumidor"]))
atendimentos["venda_tem_cliente"] = atendimentos["id_venda"].map(ref).notna()

# caminho mais fraco (fora da J1): o nome completo é igual ao de um único cliente
def chave(nome):
    return unicodedata.normalize("NFKD", " ".join(str(nome).split())).encode("ascii", "ignore").decode("ascii").lower()
nomes = clientes["nome_busca"].value_counts()
atendimentos["nome_unico"] = atendimentos["nome_contato"].dropna().map(chave).pipe(
    lambda k: k.str.contains(" ") & k.isin(nomes[nomes == 1].index)).reindex(atendimentos.index, fill_value=False)

def cliente(a):
    pelo_pedido, pelo_contato = a["cliente_pelo_pedido"], a["cliente_pelo_contato"]
    if a["ponte"]:
        return "ponte e-mail ↔ CPF no mesmo atendimento"
    if pd.notna(pelo_pedido) and pd.notna(pelo_contato):
        return "pedido e contato concordam" if pelo_pedido == pelo_contato else "pedido e contato discordam"
    if pd.notna(pelo_pedido):
        if pelo_pedido in cpfs:
            return "só pelo pedido"
        return "pelo pedido, só e-mail (cliente sem conta)" if "@" in pelo_pedido else "pelo pedido, CPF fora do cadastro"
    if pd.notna(pelo_contato):
        return "pelo contato, e-mail de comprador" if "@" in pelo_contato else "só pelo contato"
    if pd.notna(a["id_venda"]):
        return "comprador sem CPF nem e-mail" if a["venda_tem_cliente"] else "venda sem cliente"
    if a["nome_unico"]:
        return "nome igual a um cliente"
    if a["tags"] in ("pre-venda", "loja"):
        return "pré-venda (não precisa de cliente)"
    return "sem identificação"

descarte = atendimentos["assunto"].eq("spam") | primeira_fala.str.strip().str.lower().eq("teste")
migraveis = atendimentos[~descarte].copy()
migraveis["cliente"] = migraveis.apply(cliente, axis=1)
print("descartados (spam e 'teste'):", descarte.sum(), "| a migrar:", len(migraveis))
migraveis["cliente"].value_counts()

In [ ]:
# J3: cada regra liga ao pedido certo? quando o contato e o pedido têm cliente, os dois deveriam ser a mesma pessoa
ambos = atendimentos["cliente_pelo_contato"].notna() & atendimentos["cliente_pelo_pedido"].notna()
atendimentos["concorda"] = (atendimentos["cliente_pelo_contato"] == atendimentos["cliente_pelo_pedido"]).where(ambos)
display(atendimentos.groupby("como_achou_venda").agg(atendimentos=("id", "size"), conferiveis=("concorda", "count"), concordam=("concorda", "sum")))

# a janela de 60 dias do número solto: dentro e fora dela, o dono da venda é o mesmo cliente do contato?
def dias_ate_o_atendimento(atendimento):
    codigo = atendimento["codigo"]
    if pd.isna(codigo) or not codigo.isdigit() or codigo in externos.index or "APP-" + codigo in externos.index or codigo not in por_id.index:
        return None
    return (pd.to_datetime(atendimento["data_hora"]) - pd.to_datetime(por_id.loc[codigo, "criado_em"])).days

soltos = atendimentos.assign(dias=atendimentos.apply(dias_ate_o_atendimento, axis=1)).dropna(subset=["dias"])
dono = soltos["codigo"].map(vendas.set_index("id")["identidade"])
soltos["concorda"] = (soltos["cliente_pelo_contato"] == dono).where(soltos["cliente_pelo_contato"].notna() & dono.notna())
faixa = pd.cut(soltos["dias"], [-10000, -1, 60, 10000], labels=["venda depois", "até 60 dias antes", "mais de 60 dias antes"])
display(soltos.groupby(faixa, observed=False).agg(atendimentos=("id", "size"), conferiveis=("concorda", "count"), concordam=("concorda", "sum")))
repetidos = vendas.dropna(subset=["pedido_externo"])
print("atendimentos que citam um código repetido (V5):", atendimentos["codigo"].replace(r"^(7\d{6})$", r"APP-\1", regex=True)
      .isin(repetidos.loc[repetidos["pedido_externo"].duplicated(), "pedido_externo"]).sum())

In [ ]:
# clientes fora do cadastro do Vulto+: com CPF ou só com e-mail viram cliente sem conta (P2); sem os dois, ficam sem cliente
fora = ["pelo pedido, CPF fora do cadastro", "pelo pedido, só e-mail (cliente sem conta)", "pelo contato, e-mail de comprador",
        "comprador sem CPF nem e-mail"]
for categoria in fora:
    grupo = migraveis[migraveis["cliente"] == categoria]
    print(categoria, "|", len(grupo), "| tags:", grupo["tags"].fillna("-").value_counts().to_dict())
discordam = migraveis[migraveis["cliente"] == "pedido e contato discordam"]
print("discordam:", len(discordam), "| pedido por e-mail:", discordam["cliente_pelo_pedido"].str.contains("@").sum())

# nas vendas: quantas online ganham cliente com o e-mail (o banco exige cliente no pedido online)
online = vendas["loja_id"].isin(["5", "6"])
so_email = vendas["identidade"].str.contains("@", na=False)
print("vendas online:", online.sum(), "| sem identidade:", (online & vendas["identidade"].isna()).sum(),
      "| identificadas só por e-mail:", (online & so_email).sum(), "| pessoas só por e-mail:", vendas.loc[so_email, "identidade"].nunique())

In [ ]:
# os que não têm nenhum identificador: o que são?
sem = migraveis[migraveis["cliente"] == "sem identificação"]
em_aberto = sem["status"].str.lower().isin(["aberto", "pendente", "em andamento", "aguardando cliente"])
print("tags:", sem["tags"].fillna("-").value_counts().to_dict())
print("contato:", sem["contato_tipo"].fillna("sem contato").value_counts().to_dict(), "| em aberto:", em_aberto.sum())
print("troca 'comprei na loja':", sem["mensagem"].str.contains("comprei na loja", case=False).sum(),
      "| com defeito:", sem["mensagem"].str.contains("defeito", case=False).sum())
sem.groupby("tags")["mensagem"].first().str.split(r"\[").str[0].str[:120]

## Anotações para o dicionário de dados

**O que é cada linha**
- Um atendimento (1.647, de 27/02/2025 a 30/09/2026): a conversa inteira, com as respostas do atendente dentro do próprio texto (`[Atendente]: ...`) em 1.105 deles. Não há uma linha por mensagem.

**Colunas**

| Coluna | O que é | Problemas vistos |
|---|---|---|
| `canal` | WhatsApp, Instagram ou e-mail | 9 grafias (`whatsapp`, `WhatsApp`, `wpp`, `insta`, `Instagram DM`...) |
| `contato` / `contato_tipo` / `contato_valor` | quem escreveu e o valor limpo | telefone (794), Instagram (408), e-mail (321), sem contato (124) |
| `assunto` | tema | 17 grafias, 237 vazios; inclui `spam` (10) e `elogio` |
| `status` | situação | 9 grafias: concluído (`ok`, `resolvido`, `Resolvido`, `fechado`, `finalizado`) e em aberto (`em andamento`, `aguardando cliente`, `pendente`, `aberto`); 81 vazios |
| `atendente` | quem atendeu | 8 grafias para ~5 atendentes (`bot` e `Bot Vulto`; `julia` e `Júlia (SAC)`; `Renata` e `renata.sac`) |
| `tags` | etiquetas | 6 combinações (`troca;tamanho`, `entrega;atraso`, `pre-venda`, `vulto+;pontos`...) |
| `pedido` / `pedido_tipo` / `pedido_codigo` | pedido citado e o código limpo | só 678 têm pedido; site `S...`, app `APP-...` ou número solto |

**Ligação com o cliente**
- Telefone: 391 acham o cliente no cadastro, 403 não. E-mail: 198 acham, 123 não.
- **Instagram (408): não há como achar**, porque o cadastro não guarda o @.
- 124 sem contato.

**Ligação com o pedido**
- 678 com pedido: 245 acham pelo número do site/app; 329 por um número solto que bate com o id interno da venda (pode ser coincidência: ligação menos confiável); 104 não acham.

**Problemas dos atendimentos (lista única, nada decidido)**

| # | Problema | Quantos | Afeta |
|---|---|---|---|
| 1 | Instagram sem como ligar ao cliente pelo contato (o pedido resolve parte: seção 5) | 408 | atendimento ligado ao cliente |
| 2 | Telefone ou e-mail que não está no cadastro | 526 | atendimento ligado ao cliente |
| 3 | Sem pedido ou pedido que não acha a venda | 969 + 104 | atendimento ligado ao pedido |
| 4 | Número solto que pode bater com o id errado | 329 | ligação com o pedido |
| 5 | Conversa inteira num campo só | 1.105 | histórico de mensagens no nosso módulo |
| 6 | Canal, status, assunto e atendente sem padrão | todos | de-para para o nosso atendimento |
| 7 | Spam | 10 | o que migrar |
| 8 | Pedido escrito só no texto da mensagem | 282 | ligação com o pedido |
| 9 | Pedido do app sem o `APP-` e pedido com os dígitos invertidos | 89 + 43 | ligação com o pedido |
| 10 | Cliente fora do cadastro do Vulto+ (122 com CPF, 189 só com e-mail) | 311 | atendimento ligado ao cliente |
| 11 | Sem nenhum identificador (nem contato, nem pedido) | 161 | atendimento ligado ao cliente |
| 12 | Mensagem "teste" | 4 | o que migrar |

**Desafios da revisão**
- **Os números soltos parecem a venda certa:** dos 329, em 311 a venda é de até 60 dias antes do atendimento (mediana de 11 dias, parecida com os pedidos do site e do app, de 9 dias). Só 18 são suspeitos (11 com a venda depois do atendimento, 7 com mais de 60 dias). A ligação é mais confiável do que parecia; os 18 podem ficar sem pedido.

**Cruzamento cliente × pedido (seção 5)**
- O pedido citado é um segundo caminho até o cliente: o dono do pedido (`cliente_ref` da venda).
- Pedidos escondidos: 282 estão só no texto (a coluna `pedido` veio vazia); 89 são do app sem o `APP-` (64 na coluna, 25 no texto; nenhum bate com o id interno); 43 têm os dígitos invertidos (`874101` → `S101478`).
- **Os invertidos são a venda certa:** dos 47 códigos que batem invertidos, em 43 a venda é de até 60 dias antes (mediana de 9 dias, igual aos pedidos do site e do app). Os códigos do site vão de `S100232` a `S990039`: por coincidência a data não bateria.
- Com isso, 928 atendimentos acham a venda: 397 pelo código, 399 por número solto, 89 pelo app sem prefixo e 43 invertidos. 32 citam um código que não acha nada; 687 não citam pedido.
- **Conferência das regras (J3):** quando o contato e o pedido têm cliente, os dois são a mesma pessoa em todas as regras (código, número solto, app sem prefixo e invertidos). Fora da janela de 60 dias, o número solto concorda em só 5 de 7. Os 6 atendimentos que citam um código repetido (V5) vão para a linha paga, que é a que migra.

| Cliente (1.633 a migrar, sem os 10 spam e os 4 "teste"; regra da J1) | Quantos |
|---|---|
| Pedido e contato acham o mesmo cliente (335 pelo CPF, 14 pelo e-mail) | 349 |
| Só o contato acha (CPF do cadastro) | 238 |
| Só o pedido acha (CPF do cadastro) | 207 |
| Pelo pedido, CPF fora do cadastro (cliente sem conta, P2) | 100 |
| Pelo pedido, só e-mail (cliente sem conta, P2) | 161 |
| Ponte: um lado dá o CPF e o outro só o e-mail | 35 |
| Pelo contato, e-mail de quem comprou | 14 |
| Venda sem cliente (loja física sem CPF) | 69 |
| Comprador sem CPF nem e-mail (só telefone fora do cadastro) | 5 |
| Nome completo igual ao de um único cliente | 39 |
| Pré-venda (não precisa de cliente) | 255 |
| Sem nenhum identificador | 161 |

- **Com cliente: 1.104** (793 com CPF do cadastro, 122 com CPF fora dele e 189 só pelo e-mail); eram 589 só pelo contato. Sem cliente: 529, dos quais 255 são pré-venda.
- **Quando os dois caminhos existem, sempre concordam** (349, nenhum discorda). Por isso a ponte é confiável: quando um lado só tem o e-mail e o outro tem o CPF, é a mesma pessoa. São 34 pares e-mail → CPF, e as 50 vendas desses e-mails passam para o CPF (menos cliente em dobro).
- **Só pelo e-mail:** nas vendas, 1.964 online são de 1.248 pessoas identificadas só pelo e-mail; sem o e-mail, o banco recusaria esses pedidos (`online_exige_cliente`). Essas pessoas não têm pontos no Vulto+ (que só credita por CPF), então o e-mail não mexe no passivo.
- Instagram (408): 204 ganham cliente pelo pedido (120 do cadastro, 30 com CPF fora dele, 54 só pelo e-mail); 84 são pré-venda, 21 vendas sem cliente, 19 nomes iguais, 3 compradores só com telefone e 71 ficam sem nada.
- **Venda sem cliente (69):** todas de troca ou devolução, de venda da loja física sem cliente (`NAO INFORMADO`, `consumidor` ou vazio).
- **Sem nenhum identificador (161):** troca/devolução 90 (28 "comprei na loja X", 26 com defeito), Vulto+ 50 ("meus pontos sumiram", "ganhei 800 pontos numa compra de 300"), entrega 7, pagamento 6, sem tag 8; 36 em aberto. Os de Vulto+ confirmam o bug do app e os pontos que somem (`04_clientes_pontos`).

*(continuar preenchendo ao longo da análise)*